# 🚩 शिव एआई (Shiv AI) - मास्टर ट्रेनिंग फैक्ट्री
**मालिक:** श्री राम नाग 

यह नोटबुक ३२ बॉट्स की शक्ति और टर्बो स्पीड के साथ काम करती है। इसमें प्रोग्रेस बार और री-ट्रेनिंग की सुविधा जुड़ी है।

In [ ]:
# @title ⚙️ सेटअप और टर्बो लाइब्रेरीज़
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q wikipedia requests datasets
print("✅ शिव एआई के ट्रेनिंग टूल्स तैयार हैं!")

In [ ]:
# @title 📂 पुराने मॉडल को लोड करें (री-ट्रेनिंग फॉर्म)
from google.colab import drive
import os
from unsloth import FastLanguageModel

drive.mount('/content/drive')

# फॉर्म ऑप्शंस
LOAD_FROM_DRIVE = False #@param {type:"boolean"}
MODEL_PATH = "/content/drive/MyDrive/ShivAI_Final_Model" #@param {type:"string"}

if LOAD_FROM_DRIVE and os.path.exists(MODEL_PATH):
    print(f"🔄 पुराने मॉडल को {MODEL_PATH} से लोड किया जा रहा है...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = MODEL_PATH,
        max_seq_length = 2048,
        load_in_4bit = True,
    )
else:
    print("🆕 नया फ्रेश मॉडल लोड किया जा रहा है...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "HuggingFaceTB/SmolLM2-135M-Instruct",
        max_seq_length = 2048,
        load_in_4bit = True,
)
print("✅ मॉडल तैयार है!")

In [ ]:
# @title 🌐 इंटरनेट से महाज्ञान डाउनलोड करना
import wikipedia
import requests
from datasets import Dataset

wikipedia.set_lang("hi")
topics = ["मशीन लर्निंग", "कृत्रिम बुद्धिमत्ता", "ब्रह्मांड"]
kb = []

for t in topics:
    try:
        p = wikipedia.page(t)
        kb.append({"instruction": f"{t} क्या है?", "response": p.summary})
    except: pass

dataset = Dataset.from_list(kb)
print(f"✅ {len(kb)} नए विषय ट्रेनिंग के लिए तैयार हैं!")

In [ ]:
# @title 🚀 प्रोग्रेस बार के साथ ट्रेनिंग शुरू करें
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "instruction",
    args = TrainingArguments(
        max_steps = 50, 
        learning_rate = 2e-4,
        logging_steps = 1, # यह प्रोग्रेस बार दिखाएगा
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("🚩 शिव एआई की ट्रेनिंग चल रही है...")
trainer.train()
print("✅ ट्रेनिंग पूरी हुई!")

In [ ]:
# @title 📥 ड्राइव में सुरक्षित करें और डाउनलोड करें
from google.colab import files
save_dir = "/content/drive/MyDrive/ShivAI_Final_Model"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ मॉडल ड्राइव में लॉक हो गया: {save_dir}")

os.system(f"zip -r ShivAI_Backup.zip {save_dir}")
files.download('ShivAI_Backup.zip')
print("🚀 बैकअप फाइल डाउनलोड हो रही है!")